In [ ]:
%matplotlib inline
# changed from %matplotlib notebook as graphs werent being plotted 

In [ ]:
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Seq import Seq 
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import numpy as np
from numpy import random as rd
from matplotlib import pyplot as plt

In [ ]:
from src.estimalign import estimalign
from src.logit_link import logit_partial_scores
from src.optimization import create_powerstep, create_constant_step

# Data

In [ ]:
from miRBench.dataset import list_datasets, get_dataset_df

In [ ]:
hejret_train = get_dataset_df(list_datasets()[0], split="train")
hejret_test = get_dataset_df(list_datasets()[0], split="test")

In [ ]:
mirlist = hejret_train['noncodingRNA']
mirlist = [Seq(seq) for seq in mirlist]
genelist = hejret_train['gene']
genelist = [Seq(seq).reverse_complement() for seq in genelist]

# Optimization

### Simple model on miRNA alignments:

In [ ]:
true_match = 1
true_mismatch = -1
true_gapopen = -1.2
true_gapext = -0.1

In [ ]:
aligner = PairwiseAligner()
aligner.mode = 'local'
aligner.open_gap_score = true_gapopen
aligner.extend_gap_score = true_gapext
aligner.match = true_match
aligner.mismatch = true_mismatch
# aligner.end_gap_score = 0

In [ ]:
scores = np.array([aligner.score(a, b) for a, b in zip(mirlist, genelist)])

In [ ]:
plt.figure()
plt.hist(scores, bins=100)
plt.show()

In [ ]:
true_alpha = -9
## true_alpha = -np.median(scores)

In [ ]:
logit_scores = logit_partial_scores(scores, true_alpha)

In [ ]:
plt.figure()
plt.hist(logit_scores, bins=100)
plt.show()

### Test Run

In [ ]:
labels = rd.rand(len(mirlist))
labels = labels <= logit_scores
labels

In [ ]:
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
print('Sum of log-logit scores:', np.sum(np.log(logit_scores)))
print('True LogL:', true_logL) ## This is different 

In [ ]:
plt.figure()
plt.plot(scores, labels, 'r.', alpha=0.005)

In [ ]:
plt.figure()
plt.plot(logit_scores, labels, 'r.', alpha=0.005)

In [ ]:
const_step = create_constant_step(0.00001)
# powerstep = create_powerstep(0.00001, power=0.5, burnin=0)
# powerstep = create_powerstep(0.00001, power=-0.5, burnin=0)

In [ ]:
NITER = 50# original 50

In [ ]:
params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    verbose=True, max_iter=NITER,
                    stochastic_factor=None,
                    num_threads = 16)

In [ ]:
print(params['final_loglik'])

In [ ]:
print(params['final_loglik'])

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
print(true_match, params['match_score'])
print(true_mismatch, params['mismatch_score'])
print(true_gapopen, params['open_gap_score'])
print(true_gapext, params['extend_gap_score'])
print(true_alpha, params['alpha'])

### Step function parameters experiment

In [ ]:
labels = rd.rand(len(mirlist)) <= logit_scores

In [ ]:
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))

In [ ]:
steplengths = np.linspace(0.000005, 0.00005, num=10)
steplengths

In [ ]:
NITER = 200

In [ ]:
# Get the iteration at which logL is hit (speed)
def get_logL__reached_iter(params, true_logL):
    for i, logL in enumerate(params['loglik_trajectory']):
        if logL >= true_logL:
            return i
    return None

In [ ]:
# Get the value at the last iteration of the simulation
def get_final_val(params):
    return params['loglik_trajectory'][-1]

In [ ]:
estimalign_results_step = []

logL_reached_iters = []
final_vals = []

for stepl in steplengths:
    const_step = create_constant_step(stepl)
    
    params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    gap_mode = 'affine',
                    verbose=False, max_iter=NITER,
                    stochastic_factor=0.001,
                    num_threads = 16)
    estimalign_results_step.append(params)

    logL_reached_iter = get_logL__reached_iter(params, true_logL)
    logL_reached_iters.append(logL_reached_iter if logL_reached_iter is not None else np.inf) # infinity so that they dont show on the scatter plot 

    final_val = get_final_val(params)
    final_vals.append(final_val)

    if logL_reached_iter is not None:
        print(f"step {stepl:.2e}, surpassed logL at {logL_reached_iter} and reached a maximum value of {final_val}")
    else:
        print(f"step {stepl:.2e} did NOT hit logL")

In [ ]:
plt.figure()
for params in estimalign_results_step:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.5)
plt.plot([0, NITER], [true_logL, true_logL], 'k--')
plt.legend([f"{sl:.1e}" for sl in steplengths])
plt.tight_layout()
plt.savefig("results/simple/step_experiment_full_trajectories.png", dpi=160)

In [ ]:
# zoom on start
plt.figure()
for params in estimalign_results_step:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.5)
plt.plot([0, NITER], [true_logL, true_logL], 'k--')
plt.legend([f"{sl:.1e}" for sl in steplengths])
plt.xlim(0, 5)
plt.tight_layout()
plt.savefig("results/simple/step_experiment_zoom_start.png", dpi=160)

In [ ]:
# zoom where most trajectories hit logL
plt.figure()
for params in estimalign_results_step:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.5)
plt.plot([0, NITER], [true_logL, true_logL], 'k--')
plt.legend([f"{sl:.1e}" for sl in steplengths])
plt.xlim(20, 75) 
plt.ylim(3250, 3200)
plt.tight_layout()
plt.savefig("results/simple/step_experiment_zoom_mid.png", dpi=160)

In [ ]:
# zoom on the end of the iterations
plt.figure()
for params in estimalign_results_step:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.5)
plt.plot([0, NITER], [true_logL, true_logL], 'k--')
plt.legend([f"{sl:.1e}" for sl in steplengths])
plt.xlim(NITER-20, NITER) 
plt.ylim(3220, 3180)
plt.tight_layout()
plt.savefig("results/simple/step_experiment_zoom_end.png", dpi=160)

In [ ]:
plt.figure()

logL_reached_iters_arr = np.array(logL_reached_iters)
final_vals_arr = np.array(final_vals)

plt.scatter(logL_reached_iters_arr, final_vals_arr)

for i, steplength in enumerate(steplengths):
    if np.isfinite(logL_reached_iters_arr[i]):
        plt.annotate(
            f"{steplength:.2e}",
            (logL_reached_iters_arr[i], final_vals_arr[i])
        )

plt.xlabel("Iteration where true logL is first reached")
plt.ylabel("Final log-likelihood at last iteration")
plt.tight_layout()
plt.savefig("results/simple/step_experiment_scatter_hit_vs_final.png", dpi=160)

### Replicates

In [ ]:
REPS = 20 #20
NITER = 200

In [ ]:
const_step = create_constant_step(3.0e-05) # choose best step from prev exp 
# const_step = create_constant_step(0.00001)
# powerstep = create_powerstep(0.00001, power=0.5, burnin=0)
# powerstep = create_powerstep(0.00001, power=-0.5, burnin=0)

In [ ]:
# Checks if the chosen step size is stable in convergence across different datasets
estimalign_results_rep = []
true_logLs = []

logL_reached_iters_rep = []
final_vals_rep = []

for _ in range(REPS):
    labels = rd.rand(len(mirlist)) <= logit_scores
    true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
    true_logLs.append(true_logL)
    params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    gap_mode = 'affine',
                    verbose=False, max_iter=NITER,
                    stochastic_factor=0.001,
                    num_threads = 16)
    estimalign_results_rep.append(params)

    logL_reached_iter = get_logL__reached_iter(params, true_logL)
    
    logL_reached_iters_rep.append(
        logL_reached_iter if logL_reached_iter is not None else np.inf
    )

    final_vals_rep.append(get_final_val(params))

for i, it in enumerate(logL_reached_iters_rep):
    if np.isfinite(it):
        print(f"rep {i} hit logL at iteration {it}, final={final_vals_rep[i]:.2f}")
    else:
        print(f"rep {i} did NOT hit logL, final={final_vals_rep[i]:.2f}")

In [ ]:
plt.figure(figsize=(7.5, 2.1))
plt.subplot(121)
for params in estimalign_results_rep:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.2, color='b')
plt.subplot(122)
plt.plot([0, NITER], [0, 0], 'k--')
for params, tlL in zip(estimalign_results_rep, true_logLs):
    plt.plot(np.arange(NITER+1), tlL - params['loglik_trajectory'], alpha=0.2, color='b')

plt.savefig("results/simple/replicates_trajectories.png", dpi=160)